# Demand-similarity weights and shrinkage

Compare basic, original spatial, and weighted spatial features. The weighted model retains the original neighbour mean. The historical weight reference remains January–August 2024, including during the full-year estimator refit.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
from arch.bootstrap import MovingBlockBootstrap
from data_tools import *
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features_neighbor_mean.parquet"))


In [ ]:
# Estimate historical hourly means from training rows only.
weight_zone_means = df.loc[
    df["tpep_pickup_datetime"].dt.month <= 8
].groupby("PULocationID")["trip_count"].mean()

def weights(zone, neighbors, tau = 1.0):
    if not np.isfinite(tau) or tau <= 0:
        raise ValueError("tau must be finite and strictly positive")
    neighbor_ids = pd.Index(list(neighbors), name = "neighbor_zone")
    unknown = neighbor_ids.difference(weight_zone_means.index)
    if len(unknown):
        raise KeyError(f"Neighbours with no training demand mean: {unknown.tolist()}")

    mu_zone = weight_zone_means.loc[zone]
    mu_neighbors = weight_zone_means.reindex(neighbor_ids)
    log_difference = np.abs(np.log1p(mu_zone) - np.log1p(mu_neighbors))
    return np.exp(-log_difference / tau).rename("weight")

# Example: weights(2, [30, 132], tau=1.0)


In [ ]:
X_spatial = df.copy()
X_basic = df.drop(columns = ["neighbors_mean_PU"])

# Compute demand-similarity weights internally, then form the numeric signal S(z,t).
import geopandas as gpd
from pathlib import Path

tau = 1.0
lambda_reg = 0.1
weights_zones = gpd.read_file(
    project_root / "taxi_zones" / "taxi_zones.shp"
)
weights_by_zone = {}
for zone_id in X_spatial["PULocationID"].unique():
    target_geometry = weights_zones.loc[weights_zones["LocationID"] == zone_id].geometry.iloc[0]
    shared_boundaries = weights_zones.boundary.intersection(target_geometry.boundary)
    neighbor_ids = weights_zones.loc[
        (~shared_boundaries.is_empty) & (weights_zones["LocationID"] != zone_id), "LocationID"
    ]
    weights_by_zone[zone_id] = weights(zone_id, neighbor_ids, tau = tau).to_dict()

# Matrix entry (z, j) contains the unnormalised pair weight w_zj.
weight_zone_ids = pd.Index(sorted(weights_by_zone), name = "PULocationID")
weight_matrix = pd.DataFrame(0.0, index = weight_zone_ids, columns = weight_zone_ids)
for zone_id, neighbor_weights in weights_by_zone.items():
    for neighbor_id, pair_weight in neighbor_weights.items():
        weight_matrix.loc[zone_id, neighbor_id] = pair_weight
weight_values = weight_matrix.to_numpy()
weight_sums = weight_values.sum(axis = 1)
zone_has_neighbors = np.array([bool(weights_by_zone[z]) for z in weight_zone_ids])
if np.any(zone_has_neighbors & (weight_sums == 0)):
    raise ValueError("All neighbour weights underflowed for a zone; use a larger tau")

# Exact previous timestamps, including missing history at the start of 2024.
weighted_pickups = X_spatial.pivot(
    index = "tpep_pickup_datetime", columns = "PULocationID", values = "trip_count"
).sort_index().reindex(columns = weight_zone_ids)
previous_weighted_pickups = weighted_pickups.reindex(
    weighted_pickups.index - pd.Timedelta(hours = 1)
).to_numpy()

# S(z,t) = sum_j(w_zj * Y_j,t-1) / (sum_j(w_zj) + lambda_reg).
weighted_neighbor_signal = np.nan_to_num(previous_weighted_pickups, nan = 0.0) @ weight_values.T
weighted_neighbor_signal[:, zone_has_neighbors] /= weight_sums[zone_has_neighbors] + lambda_reg
missing_weighted_history = (
    np.isnan(previous_weighted_pickups).astype(float) @ (weight_values > 0).T.astype(float)
) > 0
weighted_neighbor_signal[missing_weighted_history] = np.nan
# Isolated zones have a zero numerator and retain a zero signal.
weighted_neighbor_signal[:, ~zone_has_neighbors] = 0

weighted_row_positions = weighted_pickups.index.get_indexer(X_spatial["tpep_pickup_datetime"])
weighted_col_positions = weight_zone_ids.get_indexer(X_spatial["PULocationID"])
assert (weighted_row_positions >= 0).all() and (weighted_col_positions >= 0).all()
X_weights = X_spatial.copy()
# Despite the column name, this is weighted demand in pickups/hour, not a static weight.
X_weights["weights"] = weighted_neighbor_signal[weighted_row_positions, weighted_col_positions]


In [ ]:
# Base features: January–August training, September validation, October–December test.
(
    X_train_basic, X_val_basic, X_test_basic,
    y_train_basic, y_val_basic, y_test_basic,
    train_basic, val_basic, test_basic,
) = split(X_basic)

# Same dates and rows, with the neighbour-demand feature included.
(
    X_train_spatial, X_val_spatial, X_test_spatial,
    y_train_spatial, y_val_spatial, y_test_spatial,
    train_spatial, val_spatial, test_spatial,
) = split(X_spatial)

# Same dates and rows, with the weighted neighbour-demand signal included.
(
    X_train_weights, X_val_weights, X_test_weights,
    y_train_weights, y_val_weights, y_test_weights,
    train_weights, val_weights, test_weights,
) = split(X_weights)


In [ ]:
model_basic = model(train_basic)
model_spatial = model(train_spatial)
model_weight = model(train_weights)

model_basic.fit(train_basic, y_train_basic)
model_spatial.fit(train_spatial, y_train_spatial)
model_weight.fit(train_weights, y_train_weights)


In [ ]:
y_pred_basic = model_basic.predict(val_basic)
y_pred_spatial = model_spatial.predict(val_spatial)
y_pred_weight = model_weight.predict(val_weights)

print(f"Basic:    MAE = {mean_absolute_error(y_val_basic, y_pred_basic):.4f}, RMSE = {root_mean_squared_error(y_val_basic, y_pred_basic):.4f}")
print(f"Spatial:  MAE = {mean_absolute_error(y_val_spatial, y_pred_spatial):.4f}, RMSE = {root_mean_squared_error(y_val_spatial, y_pred_spatial):.4f}")
print(f"Weighted: MAE = {mean_absolute_error(y_val_weights, y_pred_weight):.4f}, RMSE = {root_mean_squared_error(y_val_weights, y_pred_weight):.4f}")


In [ ]:
# Same thresholds as before, estimated on January–August training data only.
category_zone_means = X_train_basic.groupby("PULocationID")["trip_count"].mean()
neighbor_based_zone_groups = {
    "Low demand, busy neighbours": [],
    "Low demand, quiet neighbours": [],
    "Demand >= 1 pickup/hour": [],
    "Low demand, no neighbours": [],
}
for zone, own_mean in category_zone_means.items():
    neighbors = list(weights_by_zone[zone])
    if own_mean >= 1:
        category = "Demand >= 1 pickup/hour"
    elif not neighbors:
        category = "Low demand, no neighbours"
    elif category_zone_means.loc[neighbors].mean() >= 10:
        category = "Low demand, busy neighbours"
    else:
        category = "Low demand, quiet neighbours"
    neighbor_based_zone_groups[category].append(zone)

assert val_basic.index.equals(val_spatial.index) and val_basic.index.equals(val_weights.index)
pd.testing.assert_series_equal(y_val_basic, y_val_spatial)
pd.testing.assert_series_equal(y_val_basic, y_val_weights)
category_metric_rows = []
for category, zones in neighbor_based_zone_groups.items():
    mask = val_basic["PULocationID"].isin(zones).to_numpy()
    actual = y_val_basic.to_numpy()[mask]
    row = {"Category": category, "Zones": len(zones)}
    for name, predictions in [("Basic", y_pred_basic), ("Spatial", y_pred_spatial), ("Weighted", y_pred_weight)]:
        predicted = np.asarray(predictions)[mask]
        row[f"{name} MAE"] = mean_absolute_error(actual, predicted)
        row[f"{name} RMSE"] = root_mean_squared_error(actual, predicted)
    category_metric_rows.append(row)

category_validation_metrics = pd.DataFrame(category_metric_rows).set_index("Category")
print("September 2024 validation — categories based on January–August 2024 demand")
print("Low: <1 pickup/hour; busy neighbours: mean >=10 pickups/hour. Lower errors are better.")
print(category_validation_metrics.to_string(float_format = lambda value: f"{value:.4f}"))


In [ ]:
import joblib
from pathlib import Path

model_dir = project_root / "models"
model_dir.mkdir(exist_ok = True)

# Save the January–August fitted pipelines; the full-year refits are below.
joblib.dump(model_basic, model_dir / "xgboost_basic.joblib")
joblib.dump(model_spatial, model_dir / "xgboost_spatial.joblib")
joblib.dump(model_weight, model_dir / "xgboost_weights.joblib")


In [ ]:
df_2025 = pd.read_parquet(
    project_root / "data/yellow_taxi_2025_features_neighbor_mean.parquet"
)
X_spatial_2025 = df_2025.copy()
X_basic_2025 = df_2025.drop(columns = ["neighbors_mean_PU"])

# Include the final hour of 2024 for the first prediction hour of 2025.
first_hour = df_2025["tpep_pickup_datetime"].min()
history = pd.concat([
    df.loc[df["tpep_pickup_datetime"] == first_hour - pd.Timedelta(hours = 1)],
    df_2025,
])
pickups_2025 = history.pivot(
    index = "tpep_pickup_datetime", columns = "PULocationID", values = "trip_count"
).sort_index().reindex(columns = weight_zone_ids)
hours_2025 = pd.Index(sorted(df_2025["tpep_pickup_datetime"].unique()))
previous_pickups_2025 = pickups_2025.reindex(
    hours_2025 - pd.Timedelta(hours = 1)
).to_numpy()

# Reuse the training weights and the same shrinkage parameter.
signal_2025 = np.nan_to_num(previous_pickups_2025, nan = 0.0) @ weight_values.T
signal_2025[:, zone_has_neighbors] /= weight_sums[zone_has_neighbors] + lambda_reg
missing_history_2025 = (
    np.isnan(previous_pickups_2025).astype(float) @ (weight_values > 0).T.astype(float)
) > 0
signal_2025[missing_history_2025] = np.nan
signal_2025[:, ~zone_has_neighbors] = 0

X_weights_2025 = X_spatial_2025.copy()
X_weights_2025["weights"] = signal_2025[
    hours_2025.get_indexer(df_2025["tpep_pickup_datetime"]),
    weight_zone_ids.get_indexer(df_2025["PULocationID"]),
]
print("2025 dataset shapes:", X_basic_2025.shape, X_spatial_2025.shape, X_weights_2025.shape)


In [ ]:
y_2025 = X_basic_2025["trip_count"]

test_basic_2025 = X_basic_2025.drop(columns = ["trip_count", "tpep_pickup_datetime"])
test_spatial_2025 = X_spatial_2025.drop(columns = ["trip_count", "tpep_pickup_datetime"])
test_weights_2025 = X_weights_2025.drop(columns = ["trip_count", "tpep_pickup_datetime"])


## Refit on the whole of 2024


In [ ]:
train_basic_2024 = X_basic.drop(columns = ["trip_count", "tpep_pickup_datetime"])
train_spatial_2024 = X_spatial.drop(columns = ["trip_count", "tpep_pickup_datetime"])
train_weights_2024 = X_weights.drop(columns = ["trip_count", "tpep_pickup_datetime"])

model_basic_2024 = model(train_basic_2024).fit(train_basic_2024, X_basic["trip_count"])
model_spatial_2024 = model(train_spatial_2024).fit(train_spatial_2024, X_spatial["trip_count"])
model_weights_2024 = model(train_weights_2024).fit(train_weights_2024, X_weights["trip_count"])


In [ ]:
y_pred_2025_basic = model_basic_2024.predict(test_basic_2025)
y_pred_2025_spatial = model_spatial_2024.predict(test_spatial_2025)
y_pred_2025_weights = model_weights_2024.predict(test_weights_2025)

for name, pred in [("Basic", y_pred_2025_basic), ("Spatial", y_pred_2025_spatial), ("Weighted", y_pred_2025_weights)]:
    print(f"{name}: MAE = {mean_absolute_error(y_2025, pred):.4f}, RMSE = {root_mean_squared_error(y_2025, pred):.4f}")
